# Data Cleaning — Customer Personality Analysis

Raw Kaggle export → analysis-ready CSV. This notebook handles: missing values,
one clear outlier, feature consolidation (education/marital status tiers),
date → tenure conversion, spend/channel column renaming, and campaign-response
consolidation. Output feeds into `cp_eda.ipynb`.

## Setup

In [ ]:
import yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
with open("../config.yaml", "r") as file:
    config = yaml.safe_load(file)

data = pd.read_csv(config['data']['raw']['file8'], quotechar='"', sep="\t")
data.head(), data.shape, data.ID.nunique()

## Missing values & identifiers

In [ ]:
data.isna().sum()

Only `Income` has nulls. Drop those rows rather than impute.

In [ ]:
data = data.dropna(subset=['Income'])

In [ ]:
# ID is just a row identifier, not a feature
data = data.drop(columns=['ID'])

## Age

In [ ]:
# Dataset was collected in 2014, so age = 2014 - birth year.
data['age'] = 2014 - data.Year_Birth

data[data.age > 100]

In [ ]:
# Ages above 100 are implausible, as we see a PhD holder aged 115, it's clearly not a simple pattern. Dropping.
data = data[data.age > 100]

## Education — consolidating to 3 tiers

Raw data has 5 education categories that really only encode 3 meaningful tiers.
Collapsing them makes the ordinal encoding used later in modeling much cleaner.

In [ ]:
data.Education.value_counts()

In [ ]:
data['Education'] = data['Education'].replace({
    'Basic': 'Undergraduate',
    '2n Cycle': 'Undergraduate',
    'Graduation': 'Postgraduate',
    'Master': 'Postgraduate',
    'PhD': 'PhD',
})

## Marital status — consolidating to 3 tiers

Same idea: 8 raw categories (including data-entry noise like `'Absurd'` and
`'YOLO'`) collapsed into `Single` / `Partnered` / `Separated`.

In [ ]:
data.Marital_Status.value_counts()

In [ ]:
data['Marital_Status'] = data['Marital_Status'].replace({
    'Divorced': 'Separated',
    'Alone': 'Single',
    'Married': 'Partnered',
    'Together': 'Partnered',
    'Absurd': 'Single',
    'Widow': 'Separated',
    'YOLO': 'Single',
})

## Income outlier

In [ ]:
sns.boxplot(data=data, y='Income')
plt.title("Income — before outlier removal")
plt.show()

In [ ]:
# One extreme outlier dwarfs the rest of the distribution — drop it by value
# rather than by a hardcoded index, so this still works if upstream rows shift.
outlier_idx = data[data.Income == data.Income.max()].index
data = data.drop(outlier_idx)

## Children

In [ ]:
data['Num_Children'] = data.Kidhome + data.Teenhome
data = data.drop(columns=['Kidhome', 'Teenhome'])

In [ ]:
data['Has_child'] = data.Num_Children > 0
data.Has_child.value_counts()

## Customer tenure

In [ ]:
# Convert enrollment date into a single "days since enrollment" feature,
# anchored to a fixed baseline date (just after the dataset's collection window).
data['Dt_Customer'] = pd.to_datetime(data['Dt_Customer'], dayfirst=True, format='%d-%m-%Y')
baseline_date = pd.to_datetime('2014-07-01')
data['Customer_Tenure_in_Days'] = (baseline_date - data['Dt_Customer']).dt.days
data = data.drop(columns=['Dt_Customer'])

## Spend columns

In [ ]:
data = data.rename(columns={
    'MntWines': 'Wines',
    'MntFruits': 'Fruits',
    'MntMeatProducts': 'Meat',
    'MntFishProducts': 'Fish',
    'MntSweetProducts': 'Sweets',
    'MntGoldProds': 'Gold',
})

In [ ]:
data['total_spend'] = data['Wines'] + data['Fruits'] + data['Meat'] + data['Fish'] + data['Sweets'] + data['Gold']

In [ ]:
cols = ['total_spend', 'Wines', 'Fruits', 'Meat', 'Fish', 'Sweets', 'Gold']
sns.boxplot(data=data[cols])
plt.xticks(rotation=45)
plt.title("Spend distributions — no extreme outliers beyond the income one already removed")
plt.show()

## Purchase channel columns

In [ ]:
data = data.rename(columns={
    'NumDealsPurchases': 'Deals',
    'NumWebPurchases': 'Web',
    'NumCatalogPurchases': 'Catalogue',
    'NumStorePurchases': 'Store',
    'NumWebVisitsMonth': 'WebVisits',
})

## Campaign response columns

In [ ]:
data.nunique()

In [ ]:
# Z_Revenue / Z_CostContact are Kaggle-provided constants (same value for every
# row) used for a profit calc that isn't part of this analysis — safe to drop later.
data.Z_Revenue.unique(), data.Z_CostContact.unique()

In [ ]:
data.AcceptedCmp1.unique()

Quick look at how rare each individual campaign acceptance and the final response are, before consolidating them.

In [ ]:
n = data.shape[0]
acceptance_rates = {
    'AcceptedCmp1': data['AcceptedCmp1'].sum() / n,
    'AcceptedCmp2': data['AcceptedCmp2'].sum() / n,
    'AcceptedCmp3': data['AcceptedCmp3'].sum() / n,
    'AcceptedCmp4': data['AcceptedCmp4'].sum() / n,
    'AcceptedCmp5': data['AcceptedCmp5'].sum() / n,
    'Response':     data['Response'].sum() / n,
}
acceptance_rates

Collapse the five historical campaign flags into a single count — `Num_Camp_Success` — used later as the 'past campaign success' CRM feature.

In [ ]:
data['Num_Camp_Success'] = data[['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']].sum(axis=1)
data = data.drop(columns=['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5'])

In [ ]:
data = data.drop(columns=['Z_Revenue', 'Z_CostContact'])

## Final formatting & export

In [ ]:
data.dtypes

In [ ]:
data.columns = data.columns.map(lambda x: x.lower())

In [ ]:
data.to_csv("../data/clean/marketing_campaign.csv", index=False, encoding="utf-8", sep=";")
data.shape